In [ ]:
from pathlib import Path
import sys
_root=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code/notebook_runtime.py').is_file() or (p/'tools/notebook_runtime.py').is_file())
_helper=_root/'code' if (_root/'code/notebook_runtime.py').is_file() else _root/'tools'
sys.path.insert(0,str(_helper))
import notebook_runtime
notebook_runtime.configure(globals(), 'submission_package_reference_style_20260910/reproduction/render_strengthened.ipynb')


In [ ]:
"""Render main figures from model results and structural comparisons."""
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import render_main_figures as r

def figure1():
    table=pd.read_csv(r.D/'Figure1_joint_outcomes.csv')
    groups=['Both decrease','Origin decreases, maritime increases','Origin increases, maritime decreases','Both increase','At least one intermediate']
    colors=[r.BLUE,r.ORANGE,'#8E719E','#B87865','#CCD4D7']
    periods=[str(x) for x in range(2013,2025)]+['pooled']
    fig=plt.figure(figsize=(183/25.4,132/25.4))
    a=fig.add_axes([.12,.17,.43,.65]);b=fig.add_axes([.70,.56,.27,.26]);c=fig.add_axes([.70,.17,.27,.26])
    left=np.zeros(len(periods))
    for group,color in zip(groups,colors):
        values=table[table.outcome.eq(group)].set_index('end_year').value_share_pct.reindex(periods).to_numpy()
        assert np.isfinite(values).all()
        a.barh(np.arange(len(periods)),values,left=left,height=.70,color=color,edgecolor='white',linewidth=.25)
        left+=values
    np.testing.assert_allclose(left,100,atol=1e-9)
    names=[f'{x-1}–{str(x)[-2:]}' for x in range(2013,2025)]+['Pooled']
    a.set(yticks=range(13),yticklabels=names,ylim=(12.7,-.7),xlim=(0,100),xticks=[0,25,50,75,100],xlabel='End-year value share (%)')
    a.axhline(11.5,color=r.MUTED,lw=.6)
    r.label(a,'a')
    for ax,metric,letter,color,unit in [(b,'delta_origin_hhi','b',r.BLUE,'Minimum origin-concentration\nincrease (index difference)'),(c,'delta_top_chokepoint_share','c',r.ORANGE,'Minimum maritime-exposure\nincrease (percentage points)')]:
        e=pd.read_csv(r.D/f'Figure1{letter}_exceedance.csv')
        ax.step(e.threshold,e.value_share_pct,where='post',color=color)
        ax.set(ylim=(0,50),yticks=[0,25,50],ylabel='Value share (%)',xlabel=unit)
        if letter=='b':ax.set(ylim=(0,4),yticks=[0,2,4])
        ax.set_xlim(0,1 if letter=='b' else 100)
        ax.set_xticks([0,.5,1] if letter=='b' else [0,50,100])
        ax.axvline(.025 if letter=='b' else 2.5,color=r.MUTED,ls='--',lw=.6)
        ax.grid(axis='y',color=r.GRID,lw=.5,ls='--');r.label(ax,letter)
    fig.legend([Patch(facecolor=x) for x in colors],groups,loc='upper center',bbox_to_anchor=(.53,.98),ncol=2,columnspacing=1.5,fontsize=6)
    r.save(fig,1,[a,b,c],['a','b','c'],cols=[['b','c']])

def supplementary():
    d=pd.read_csv(r.D/'SupplementaryFigure1_preferences.csv')
    fig,axes=plt.subplots(1,2,figsize=(183/25.4,85/25.4))
    fig.subplots_adjust(left=.12,right=.98,bottom=.22,top=.82,wspace=.4)
    for j,(prefix,vals) in enumerate([('origin',[0,.25,.5,1,2,4]),('route',[.25,.5,1,2,4])]):
        ax=axes[j]
        for metal,marker in zip(['Copper','Aluminium','PGM','Nickel'],['o','s','D','^']):
            s=d[d.metal.eq(metal)].set_index('configuration')
            names=['reference' if x==1 else f'{prefix}_{x:g}' for x in vals]
            ax.plot(range(len(vals)),100*s.loc[names,'material_joint_reduction'],marker=marker,color=r.MC[metal],ms=3,label=metal)
        ax.set(xticks=range(len(vals)),xticklabels=[str(x) for x in vals],ylim=(0,100),yticks=[0,25,50,75,100],xlabel=f'{prefix.capitalize()}-term coefficient multiplier',ylabel='Three-indicator joint reduction\n(value share, %)')
        ax.grid(axis='y',color=r.GRID,lw=.5,ls='--');r.label(ax,chr(97+j))
    fig.legend(*axes[0].get_legend_handles_labels(),loc='upper center',ncol=4,bbox_to_anchor=(.55,.98))
    r.save(fig,'S1',list(axes),['a','b'],rows=[['a','b']])

if __name__=='__main__':
    figure1()
    for n in [2,3,4,5]:getattr(r,f'figure{n}')()
    supplementary()
